In [1]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

In [2]:
from src.pipeline.config import (
    TRAIN_MERGED_PATH,
	PROCESSED_DATA_DIR,
	RANDOM_SEED
)

In [3]:
df = pd.read_parquet(TRAIN_MERGED_PATH, engine="pyarrow")

In [4]:
df["TransactionDT"].is_monotonic_increasing

True

In [5]:
df["card2"] = df["card2"].fillna(-1)
df["addr1"] = df["addr1"].fillna(-1)

In [6]:
df["uid"] = df["card1"].astype(str) + "_" + df["card2"].astype("Int64").astype(str) + "_" + df["addr1"].astype("Int64").astype(str)

In [7]:
df["uid"].head(15)

0      13926_-1_315
1      2755_404_325
2      4663_490_330
3     18132_567_476
4      4497_514_420
5      5937_555_272
6     12308_360_126
7     12695_490_325
8      2803_100_337
9     17399_111_204
10     16496_352_-1
11      4461_375_-1
12     3786_418_204
13    12866_303_330
14    11839_490_226
Name: uid, dtype: str

In [8]:
df["TransactionAmtLog"] = np.log1p(df["TransactionAmt"])
df["TransactionAmtMean"] = df.groupby("uid")["TransactionAmt"].transform("mean")
df["TransactionAmtMax"] = df.groupby("uid")["TransactionAmt"].transform("max")
df["TransactionAmtMin"] = df.groupby("uid")["TransactionAmt"].transform("min")
df["TransactionAmtStd"] = df.groupby("uid")["TransactionAmt"].transform("std").fillna(0)


In [9]:
df["TransactionAmt_Z"] = (df["TransactionAmt"] - df["TransactionAmtMean"]) / (df["TransactionAmtStd"] + 1e-5)
df["TransactionAmt_to_Mean"] = df["TransactionAmt"] / (df["TransactionAmtMean"] + 1e-5)

In [10]:
df["Hour"] = (df["TransactionDT"] // 3600) % 24
df["DayOfWeek"] = (df["TransactionDT"] // 86400) % 7

df["HourSin"] = np.sin(2 * np.pi * df["Hour"] / 24)
df["HourCos"] = np.cos(2 * np.pi * df["Hour"] / 24)

df["DayOfWeekSin"] = np.sin(2 * np.pi * df["DayOfWeek"] / 7)
df["DayOfWeekCos"] = np.cos(2 * np.pi * df["DayOfWeek"] / 7)

In [11]:
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
corr_matrix = corr_matrix = df[numeric_cols].sample(100_000, random_state=RANDOM_SEED).corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
to_drop

['TransactionDT',
 'C2',
 'C4',
 'C6',
 'C8',
 'C10',
 'C11',
 'C12',
 'C14',
 'D2',
 'D6',
 'D7',
 'D12',
 'V11',
 'V16',
 'V18',
 'V21',
 'V22',
 'V28',
 'V30',
 'V32',
 'V33',
 'V34',
 'V40',
 'V43',
 'V49',
 'V50',
 'V52',
 'V57',
 'V58',
 'V60',
 'V63',
 'V70',
 'V71',
 'V72',
 'V74',
 'V81',
 'V84',
 'V88',
 'V89',
 'V91',
 'V92',
 'V93',
 'V94',
 'V97',
 'V101',
 'V102',
 'V103',
 'V106',
 'V126',
 'V127',
 'V128',
 'V132',
 'V133',
 'V134',
 'V143',
 'V145',
 'V150',
 'V151',
 'V154',
 'V155',
 'V156',
 'V159',
 'V160',
 'V163',
 'V164',
 'V167',
 'V168',
 'V177',
 'V178',
 'V179',
 'V182',
 'V192',
 'V193',
 'V196',
 'V202',
 'V204',
 'V211',
 'V212',
 'V213',
 'V216',
 'V217',
 'V218',
 'V219',
 'V222',
 'V225',
 'V231',
 'V232',
 'V233',
 'V235',
 'V236',
 'V237',
 'V244',
 'V249',
 'V251',
 'V253',
 'V254',
 'V256',
 'V263',
 'V265',
 'V266',
 'V269',
 'V272',
 'V273',
 'V275',
 'V276',
 'V277',
 'V278',
 'V279',
 'V280',
 'V292',
 'V293',
 'V294',
 'V295',
 'V296',
 'V298'

In [12]:
processed_path = PROCESSED_DATA_DIR / "final_train_df.parquet"
df.drop(to_drop, axis=1).to_parquet(processed_path, engine="pyarrow")